# Electricity Uncertainty Calibration

## Role
This notebook evaluates the availability and calibration of probabilistic uncertainty
evidence for the final Electricity model set. It carries forward the Uncertainty
Calibration evidence previously part of `15_Electricity_Trustworthiness_Evidence.ipynb`
into its own single-purpose notebook.

## Inputs
The frozen aggregate uncertainty evidence artifact (`uncertainty_summary.csv`).

## Outputs
A native-probabilistic-evidence coverage/width table, an evidence-availability record
distinguishing available from unavailable models, and a validation/audit table local to
this evidence.

## Depends On
`15_Electricity_Robustness.ipynb`

## Authoritative Status
`AUTHORITATIVE UNCERTAINTY CALIBRATION EVIDENCE`

## What This Notebook Does Not Do
This notebook does NOT:
- evaluate robustness (`15_Electricity_Robustness.ipynb`)
- compute Trust Scores (`17_Electricity_Trustworthiness.ipynb`)
- perform statistical-significance testing (`18_Electricity_Statistical_Significance.ipynb`)
- generate new probabilistic forecasts

## 1. Objective

Only Chronos-Bolt-Tiny and TimesFM provide native probabilistic interval evidence in the
current Electricity experiment.

> How well calibrated is the available native uncertainty evidence, and how should the
> absence of probabilistic evidence for the remaining models be represented without
> treating missing evidence as equivalent to bad evidence?

## 2. Setup

Only the frozen aggregate uncertainty evidence artifact is loaded. No forecasting model
is loaded, no checkpoint is downloaded, and no new probabilistic forecast is generated.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing src/")

ROOT = find_project_root(Path.cwd())
R = ROOT / "results/electricity"
SCALE = 117.057971280678

uncertainty = pd.read_csv(R / "uncertainty_summary.csv")
assert uncertainty.shape == (26, 9)

## 3. Data and Method

### 3.1 Available Evidence Types

**Native probabilistic intervals** -- Chronos-Bolt-Tiny and TimesFM are the only two
models with native 80% interval evidence recorded in the frozen artifact
(`Evidence_Type == "Native probabilistic"`).

**Unavailable uncertainty evidence** -- the remaining eleven models (Naive,
Daily_Seasonal_Naive, Weekly_Seasonal_Naive, Moving_Average, ARIMA, SARIMA, Prophet,
Simple_Exponential_Smoothing, Holt_Winters, DHR_ARIMA, LSTM) have no defensible
probabilistic interval evidence preserved and are recorded as `Unavailable`, never as a
fabricated or zero-valued interval.

> No final-test residuals may be used to manufacture post-hoc uncertainty intervals.

The artifact's own `Notes` column confirms this for every unavailable row: "No
sufficient saved pre-test validation residual evidence; final-test residuals not used."

### 3.2 Comparability Principle

Coverage and width are interpreted according to the type of uncertainty evidence
available -- both available rows share the same `Native probabilistic` evidence type and
the same 80% nominal level, so they are directly comparable to each other. Missing
evidence for the other eleven models is a **scope limitation**, not a calibration score
of zero or a statement that those models would calibrate poorly if evidence existed.

## 4. Results — Foundation-Model Calibration

### 4.1 Coverage and Width — Table

In [2]:
available = uncertainty[uncertainty.Available].copy()
display(available)
print("Aggregate Phase 5 evidence only; exact quantile vectors were not saved. "
      "Horizon-specific vectors and 95% intervals are unavailable.")

,Protocol,Model,Interval,Nominal_Coverage,Empirical_Coverage,Average_Width,Evidence_Type,Available,Notes
6,A,Chronos_Bolt_Tiny,80%,0.8,0.911231,137.419281,Native probabilistic,True,Exact aggregate Phase 5 result; horizon-specif...
7,A,TimesFM,80%,0.8,0.336495,17.108297,Native probabilistic,True,Exact aggregate Phase 5 result; horizon-specif...
14,B,Chronos_Bolt_Tiny,80%,0.8,0.676239,283.997589,Native probabilistic,True,Exact aggregate Phase 5 result; horizon-specif...
15,B,TimesFM,80%,0.8,0.245604,75.642609,Native probabilistic,True,Exact aggregate Phase 5 result; horizon-specif...


Aggregate Phase 5 evidence only; exact quantile vectors were not saved. Horizon-specific vectors and 95% intervals are unavailable.


No horizon-resolved calibration table/visualization or additional calibration metrics
(ACE, Winkler score) are present in the source evidence for Electricity uncertainty, so
none is added here -- this is a structural reorganisation of existing evidence, not new
analysis.

## 5. Results — Evidence Availability

### 5.1 Uncertainty Evidence Availability — Table

In [3]:
unavailable = uncertainty[~uncertainty.Available].copy()
display(unavailable)
assert unavailable["Notes"].str.contains("final-test residuals not used").all()

,Protocol,Model,Interval,Nominal_Coverage,Empirical_Coverage,Average_Width,Evidence_Type,Available,Notes
0,A,Naive,Unavailable,NaN,NaN,NaN,Unavailable,False,No sufficient saved pre-test validation residu...
1,A,Daily_Seasonal_Naive,Unavailable,NaN,NaN,NaN,Unavailable,False,No sufficient saved pre-test validation residu...
2,A,Weekly_Seasonal_Naive,Unavailable,NaN,NaN,NaN,Unavailable,False,No sufficient saved pre-test validation residu...
3,A,Moving_Average,Unavailable,NaN,NaN,NaN,Unavailable,False,No sufficient saved pre-test validation residu...
4,A,DHR_ARIMA,Unavailable,NaN,NaN,NaN,Unavailable,False,No sufficient saved pre-test validation residu...
5,A,LSTM,Unavailable,NaN,NaN,NaN,Unavailable,False,No sufficient saved pre-test validation residu...
8,B,Naive,Unavailable,NaN,NaN,NaN,Unavailable,False,No sufficient saved pre-test validation residu...
9,B,Daily_Seasonal_Naive,Unavailable,NaN,NaN,NaN,Unavailable,False,No sufficient saved pre-test validation residu...
10,B,Weekly_Seasonal_Naive,Unavailable,NaN,NaN,NaN,Unavailable,False,No sufficient saved pre-test validation residu...
11,B,Moving_Average,Unavailable,NaN,NaN,NaN,Unavailable,False,No sufficient saved pre-test validation residu...


## 6. Validation / Audit

Checks local to this notebook's own uncertainty evidence only -- not a repository-wide
artifact verifier.

In [4]:
rows = []
rows.append({"Category": "Artifact Loading", "Check": "uncertainty artifact loaded successfully",
             "Expected": True, "Observed": True, "Result": "PASS"})

interval_cols = {"Nominal_Coverage", "Empirical_Coverage", "Average_Width"}.issubset(uncertainty.columns)
rows.append({"Category": "Structural", "Check": "coverage/width columns exist where expected",
             "Expected": True, "Observed": bool(interval_cols), "Result": "PASS" if interval_cols else "FAIL"})

coverage_valid = bool(available.Empirical_Coverage.between(0, 1).all() and np.isfinite(available.Empirical_Coverage).all())
rows.append({"Category": "Structural", "Check": "empirical coverage values finite and within [0,1] (available rows)",
             "Expected": True, "Observed": coverage_valid, "Result": "PASS" if coverage_valid else "FAIL"})

width_valid = bool((available.Average_Width >= 0).all() and np.isfinite(available.Average_Width).all())
rows.append({"Category": "Structural", "Check": "average width finite and non-negative (available rows)",
             "Expected": True, "Observed": width_valid, "Result": "PASS" if width_valid else "FAIL"})

nominal_consistent = bool((available.Nominal_Coverage == 0.8).all())
rows.append({"Category": "Structural", "Check": "nominal coverage recorded consistently (80%) for available rows",
             "Expected": True, "Observed": nominal_consistent, "Result": "PASS" if nominal_consistent else "FAIL"})

no_calibration_leakage = bool(unavailable["Notes"].str.contains("final-test residuals not used").all())
rows.append({"Category": "Leakage Discipline", "Check": "calibration did not use final-test residuals (unavailable rows' own notes)",
             "Expected": True, "Observed": no_calibration_leakage, "Result": "PASS" if no_calibration_leakage else "FAIL"})

unavailable_explicit = bool((unavailable.Evidence_Type == "Unavailable").all() and unavailable[["Nominal_Coverage", "Empirical_Coverage", "Average_Width"]].isna().all().all())
rows.append({"Category": "Leakage Discipline", "Check": "unavailable evidence remains explicitly unavailable (not fabricated)",
             "Expected": True, "Observed": unavailable_explicit, "Result": "PASS" if unavailable_explicit else "FAIL"})

row_count_ok = len(available) == 4 and len(unavailable) == 22 and len(uncertainty) == 26
rows.append({"Category": "Structural", "Check": "2 models with evidence x 2 protocols = 4 available; 11 models x 2 protocols = 22 unavailable",
             "Expected": True, "Observed": row_count_ok, "Result": "PASS" if row_count_ok else "FAIL"})

native_label_ok = bool((available.Evidence_Type == "Native probabilistic").all())
rows.append({"Category": "Structural", "Check": "native foundation-model intervals clearly labelled (Evidence_Type == 'Native probabilistic')",
             "Expected": True, "Observed": native_label_ok, "Result": "PASS" if native_label_ok else "FAIL"})

rows.append({"Category": "MASE Denominator", "Check": "MASE denominator constant remains 117.057971280678 (not used in this notebook's own calibration metrics)",
             "Expected": SCALE, "Observed": SCALE, "Result": "PASS" if np.isclose(SCALE, 117.057971280678) else "FAIL"})

audit = pd.DataFrame(rows, columns=["Category", "Check", "Expected", "Observed", "Result"])
display(audit)
assert audit["Result"].eq("PASS").all()
print(f"ALL {len(audit)} UNCERTAINTY CALIBRATION VALIDATION CHECKS PASS")

,Category,Check,Expected,Observed,Result
0,Artifact Loading,uncertainty artifact loaded successfully,True,True,PASS
1,Structural,coverage/width columns exist where expected,True,True,PASS
2,Structural,empirical coverage values finite and within [0...,True,True,PASS
3,Structural,average width finite and non-negative (availab...,True,True,PASS
4,Structural,nominal coverage recorded consistently (80%) f...,True,True,PASS
5,Leakage Discipline,calibration did not use final-test residuals (...,True,True,PASS
6,Leakage Discipline,unavailable evidence remains explicitly unavai...,True,True,PASS
7,Structural,2 models with evidence x 2 protocols = 4 avail...,True,True,PASS
8,Structural,native foundation-model intervals clearly labe...,True,True,PASS
9,MASE Denominator,MASE denominator constant remains 117.05797128...,117.057971,117.057971,PASS


ALL 10 UNCERTAINTY CALIBRATION VALIDATION CHECKS PASS


## 7. Key Findings

In [5]:
chronos = available[available.Model == "Chronos_Bolt_Tiny"].set_index("Protocol")
timesfm = available[available.Model == "TimesFM"].set_index("Protocol")
coverage_comparison = pd.DataFrame({
    "Chronos_Bolt_Tiny Empirical Coverage": chronos.Empirical_Coverage,
    "TimesFM Empirical Coverage": timesfm.Empirical_Coverage,
    "Chronos_Bolt_Tiny |Coverage - Nominal|": (chronos.Empirical_Coverage - chronos.Nominal_Coverage).abs(),
    "TimesFM |Coverage - Nominal|": (timesfm.Empirical_Coverage - timesfm.Nominal_Coverage).abs(),
    "Chronos_Bolt_Tiny Average Width": chronos.Average_Width,
    "TimesFM Average Width": timesfm.Average_Width,
})
display(coverage_comparison)
closer_to_nominal = coverage_comparison[["Chronos_Bolt_Tiny |Coverage - Nominal|", "TimesFM |Coverage - Nominal|"]].idxmin(axis=1)
display(closer_to_nominal.to_frame("Closer to Nominal Coverage"))

,Chronos_Bolt_Tiny Empirical Coverage,TimesFM Empirical Coverage,Chronos_Bolt_Tiny |Coverage - Nominal|,TimesFM |Coverage - Nominal|,Chronos_Bolt_Tiny Average Width,TimesFM Average Width
Protocol,,,,,,
A,0.911231,0.336495,0.111231,0.463505,137.419281,17.108297
B,0.676239,0.245604,0.123761,0.554396,283.997589,75.642609


,Closer to Nominal Coverage
Protocol,
A,Chronos_Bolt_Tiny |Coverage - Nominal|
B,Chronos_Bolt_Tiny |Coverage - Nominal|


The table above reports each model's empirical coverage against the shared 80% nominal
level and which one sits closer to nominal, per protocol -- read directly from the
computed columns rather than restated as a fixed claim here, since which model is closer
can differ by protocol. Average interval width is reported alongside coverage because a
narrower interval is not automatically better evidence: a narrow but badly under-covering
interval is worse evidence than a wider, better-calibrated one (Section 3.2). Point
accuracy and calibration are distinct dimensions -- a model's aggregate MASE-48 ranking
(established in `14_Electricity_Model_Validation_Audit.ipynb` /
`17_Electricity_Trustworthiness.ipynb`) is not evaluated in this notebook and is not
assumed to predict which of the two models calibrates closer to nominal. No claim of
universal probabilistic superiority is made for either model.

## 8. Limitations

### Heterogeneous and Incomplete Evidence
Only 2 of the 13 final models (Chronos-Bolt-Tiny, TimesFM) have any uncertainty
evidence at all; the other 11 are recorded as `Unavailable`, not as poor calibration.

### One Nominal Level, No Retained Quantile Vectors
Both available rows carry evidence only at the 80% nominal level -- **95% intervals are
unavailable for every model, including Chronos-Bolt-Tiny and TimesFM** (confirmed from
the artifact's own `Notes` column: "95% unavailable"). Exact per-day/per-horizon
quantile vectors were **not saved** for either model (`Notes`: "horizon-specific vectors
not saved") -- only the aggregate scalar coverage and width are available, which limits
this notebook to marginal, whole-test-period coverage rather than any horizon-resolved
or conditional calibration statement.

### Marginal Coverage Only
Coverage figures in this notebook are marginal across the full frozen test set; no
regime- or horizon-conditional coverage is computed here.

### Missing Evidence Is Not Bad Evidence
The absence of uncertainty evidence for 11 of 13 models is a scope limitation of what
was preserved, not a statement that those models would calibrate poorly if intervals had
been generated and saved for them.

## 9. Next Notebook

Next: `17_Electricity_Trustworthiness.ipynb`